# 🧩 01x — Template pipeline (à copier pour une nouvelle source)

**Mode d'emploi :**

1. Dupliquer ce fichier sous un nouveau nom, ex :
   `cp notebooks/01x_pipeline_template.ipynb notebooks/01j_pipeline_<nom_source>.ipynb`
   (prendre la lettre suivante dans `01a`...`01i`).
2. **Rechercher/remplacer** `ma_source` par le nom réel de la source (ex :
   `meteo`, `pollen`...) **dans tout le fichier** — ce même mot sert
   à la fois de valeur pour `NOM_SOURCE`, de nom de fonction
   (`build_dim_ma_source`) et de nom de variable (`dim_ma_source`), donc un
   seul remplacement global les synchronise tous. Adapter aussi le chemin de
   `fpath` vers le vrai fichier brut.
3. Déposer le(s) fichier(s) brut(s) dans `data/raw/<nom_source>/`.
4. Compléter `build_dim_XXX()` en suivant les étapes commentées (`TODO`).
5. Exécuter tout le notebook : ça doit produire `data/processed/dim_<nom_source>.parquet`.

**Les deux seules contraintes pour que `02_merge_final.ipynb` récupère votre table
automatiquement (aucune autre modification nécessaire ailleurs) :**

- le fichier s'appelle `dim_<nom_source>.parquet` et se trouve dans `data/processed/` ;
- il contient une colonne `dept` (valeurs dans `DEPTS`, cf. `src/config.py`), et
  éventuellement `annee_mois` (format `"YYYY-MM"`, table mensuelle) ou `annee`
  (int, table annuelle — sera diffusée sur les 12 mois de l'année lors de la
  fusion). Une table sans dimension temporelle (`dept` seul, ex : `dim_csp`)
  est aussi acceptée.

`00_config_commun.ipynb` doit avoir été exécuté au moins une fois avant (il
fournit `dim_temps.parquet` et `fact_urgences.parquet` et la config partagée). Les autres `01x_pipeline_*`
n'ont pas besoin d'être lancés — chaque table est construite indépendamment.

In [ ]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


---
## 1. Source à documenter

TODO : remplacer par la vraie source.

| | |
|---|---|
| Organisme | *ex : Météo France, INSEE, DREES...* |
| URL | *lien vers le jeu de données* |
| Fichier attendu | `data/raw/<nom_source>/<fichier>` |
| Maille | *dept ? dept × mois ? dept × année ?* |

| Colonne produite | Description |
|---|---|
| `dept` | Code département (clé) |
| `annee_mois` ou `annee` | Clé temporelle (si pertinente) |
| `NOM_SOURCE_xxx` | *à compléter* |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# À PERSONNALISER
# ══════════════════════════════════════════════════════════════════════════════
NOM_SOURCE = "ma_source"  # TODO: ex "meteo", "pollution_sonore"... -> sauvegardé en dim_<NOM_SOURCE>.parquet
fpath = RAW_DIR / "ma_source" / "fichier.csv"  # TODO: chemin du fichier brut

In [ ]:
# Aperçu des colonnes brutes, pour repérer les noms/positions à utiliser ci-dessous
if fpath.exists():
    apercu = pd.read_csv(fpath, sep=",", nrows=5, dtype=str)  # TODO: séparateur (";" fréquent pour les sources FR)
    print(f"Colonnes de {fpath.name} :")
    for col in apercu.columns:
        print(f"  '{col}'")
    display(apercu)
else:
    print(f"⚠️  Fichier manquant : {fpath}")
    print("   Déposez le fichier brut au bon endroit, puis relancez cette cellule.")

---
## 2. Construction de `dim_<NOM_SOURCE>`

In [ ]:
def build_dim_ma_source() -> pd.DataFrame:
    """
    TODO : changer le nom de la fonction : build_dim_ma_source() → build_dim_<NOM_SOURCE>()
    """
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    # ── ÉTAPE 1 : Chargement ─────────────────────────────────────────────────
    df = pd.read_csv(fpath, sep=",", encoding="utf-8", dtype=str, low_memory=False)
    print(f"  Brut : {len(df):,} lignes")

     # ── ÉTAPES SUIVANTES : Nettoyage et transformation (Écrites par l'utilisateur) ─────
    
    return df

    # TODO : compléter build_dim_ma_source() avant utilisation"
dim_ma_source = build_dim_ma_source()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION + SAUVEGARDE
# ══════════════════════════════════════════════════════════════════════════════
if not dim_ma_source.empty:
    cle = ["dept", "annee_mois"]  # TODO: adapter selon la clé retenue à l'ÉTAPE 6 (["dept","annee"] ou ["dept"])
    ok = valider_dim_table(dim_ma_source, f"dim_{NOM_SOURCE}", cle=cle)
    dim_ma_source.to_parquet(TABLES_DIR / f"dim_{NOM_SOURCE}.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_{NOM_SOURCE}.parquet")
    display(dim_ma_source.head(10))

---
## Checklist avant de partager sa table

1. `valider_dim_table` n'affiche aucun `❌` (codes dept valides, pas de doublon sur la clé)
2. Fichier sauvegardé : `data/processed/dim_<nom_source>.parquet`
3. Renommer ce notebook (`01j_pipeline_<nom_source>.ipynb`, prochaine lettre libre)
4. Ajouter une ligne dans le tableau `notebooks/` du `README.md` (optionnel mais utile)
5. Relancer `02_merge_final.ipynb` pour vérifier que la table s'intègre bien à `df_model`
